In [2]:
!git clone https://github.com/gdmarmerola/cfml_tools.git
import sys
import os

import sys
import os

# Em vez de adicionar base_dir, vamos adicionar o caminho direto onde o pacote vive
repo_path = os.path.join(os.getcwd(), 'cfml_tools')
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Tenta importar novamente
from cfml_tools.tree import DecisionTreeCounterfactual
print("Importação bem-sucedida!")

fatal: destination path 'cfml_tools' already exists and is not an empty directory.
Importação bem-sucedida!


In [3]:
import sys
import os
import gc
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ============================================
# 0. SETUP E IMPORTAÇÃO
# ============================================
base_dir = os.getcwd()
repo_dir = os.path.join(base_dir, 'cfml_tools')
if not os.path.exists(repo_dir):
    os.system(f"git clone https://github.com/gdmarmerola/cfml_tools.git {repo_dir}")
if base_dir not in sys.path: sys.path.insert(0, base_dir)

from cfml_tools.tree import DecisionTreeCounterfactual
warnings.filterwarnings('ignore')

# ============================================
# 1. CONFIGURAÇÃO
# ============================================
TARGETS = ['ABE_ESP', 'ABC_ESP', 'ABB_ESP', 'ABI_ESP']
TREATMENT = 'ADITIVO'
COVARS = ['REGIAO', 'ESPÉCIE', 'FASE', 'AREA', 'ABH_ESP']
df = pd.read_csv('/usr/app/Embrapa/embrapa.csv')

# ============================================
# 2. PREPARAÇÃO E PIPELINE
# ============================================
COVARS_CAT = [c for c in COVARS if df[c].dtype == 'object' or df[c].nunique() <= 20]
COVARS_NUM = [c for c in COVARS if c not in COVARS_CAT]

df[TREATMENT] = df[TREATMENT].astype('category')
W_full = df[TREATMENT].cat.codes.values

preprocessor = ColumnTransformer(transformers=[
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), COVARS_CAT),
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), COVARS_NUM)
])

# ============================================
# 3. PROPENSITY E OVERLAP
# ============================================
X_full = preprocessor.fit_transform(df[COVARS_CAT + COVARS_NUM])
ps_model = LogisticRegression(multi_class='multinomial', max_iter=500).fit(X_full, W_full)
mask_overlap = (ps_model.predict_proba(X_full) > 0.1).sum(axis=1) >= 2
df_overlap = df[mask_overlap].copy()
W_overlap = W_full[mask_overlap]
X_overlap = preprocessor.transform(df_overlap[COVARS_CAT + COVARS_NUM])

# ============================================
# 4. FUNÇÃO DE DIAGNÓSTICO E RODAGEM
# ============================================
def rodar_y(y_name):
    print(f"\n--- Diagnóstico para {y_name} ---")
    y_vals = df_overlap[y_name].fillna(df_overlap[y_name].median()).values
    
    feature_names = preprocessor.get_feature_names_out()
    X_df = pd.DataFrame(X_overlap, columns=feature_names)
    
    # Filtro reduzido para não descartar efeitos sutis
    dtcf = DecisionTreeCounterfactual(min_sample_effect=5, save_explanatory=True)
    dtcf.fit(X_df, W_overlap, y_vals)
    cf = pd.DataFrame(dtcf.predict(X_df), columns=[f'pred_W{i}' for i in range(len(np.unique(W_overlap)))])
    
    print("Médias das predições:", cf.mean().values)
    diff = cf.max(axis=1) - cf.min(axis=1)
    print(f"Diferença média entre melhor/pior aditivo: {diff.mean():.6f}")
    
    effects = cf.sub(cf['pred_W0'], axis=0)
    
    df_res = df_overlap[COVARS + [y_name]].copy()
    col_vencedora = effects.idxmax(axis=1)
    df_res['melhor_aditivo'] = pd.to_numeric(col_vencedora.apply(lambda x: str(x).replace('pred_W', '')), errors='coerce').fillna(0).astype(int)
    df_res['ganho_max'] = effects.max(axis=1).fillna(0)
    df_res['y'] = y_name
    return df_res

# Execução
all_effects = [rodar_y(y) for y in TARGETS]
df_final = pd.concat(all_effects, ignore_index=True)
df_final.to_csv('resultado_debug.csv', index=False)

print("\nProcessamento finalizado. Verifique a 'Diferença média' acima.")


--- Diagnóstico para ABE_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

--- Diagnóstico para ABC_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

--- Diagnóstico para ABB_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

--- Diagnóstico para ABI_ESP ---
Médias das predições: [nan nan nan nan nan nan]
Diferença média entre melhor/pior aditivo: nan

Processamento finalizado. Verifique a 'Diferença média' acima.


In [8]:
import pandas as pd
import statsmodels.formula.api as smf

# 1. Definições
TARGETS = ['ABE_ESP', 'ABC_ESP', 'ABB_ESP', 'ABI_ESP']

# 2. Função de análise e estruturação
def rodar_analise_completa(df, targets):
    resultados_finais = []
    
    for y in targets:
        print(f"Processando {y}...")
        # Executa a regressão OLS
        formula = f"{y} ~ C(ADITIVO) + REGIAO + ESPÉCIE + FASE + AREA"
        model = smf.ols(formula, data=df).fit()
        
        # Extrai a tabela de coeficientes
        df_res = model.summary2().tables[1]
        
        # Filtra apenas linhas que contêm 'ADITIVO'
        df_aditivos = df_res[df_res.index.str.contains('C\(ADITIVO\)')].copy()
        
        # Adiciona ao relatório apenas os que possuem significância estatística (P < 0.05)
        for index, row in df_aditivos.iterrows():
            if row['P>|t|'] < 0.05:
                nome_aditivo = index.replace('C(ADITIVO)[T.', '').replace(']', '')
                resultados_finais.append({
                    'Alvo (y)': y,
                    'Aditivo': nome_aditivo,
                    'Coeficiente (Ganho/Perda)': round(row['Coef.'], 4),
                    'P-Valor': round(row['P>|t|'], 4),
                    'Tendência': 'Positiva' if row['Coef.'] > 0 else 'Negativa'
                })
                
    return pd.DataFrame(resultados_finais)

# 3. Execução
relatorio_df = rodar_analise_completa(df, TARGETS)

# 4. Exibição e Exportação
print("\n--- RESUMO DE ADITIVOS ESTATISTICAMENTE SIGNIFICATIVOS ---")
print(relatorio_df)


# 1. Ordenar o relatório pelos maiores coeficientes positivos
relatorio_ordenado = relatorio_df.sort_values(by=['Alvo (y)', 'Coeficiente (Ganho/Perda)'], ascending=[True, False])

# 2. Selecionar o Top 1 para cada Alvo (o maior ganho)
top_ganhos_por_y = relatorio_ordenado.groupby('Alvo (y)').head(1)

print("\n--- TOP ADITIVOS POR ALVO (MAIORES GANHOS) ---")
print(top_ganhos_por_y)

# 3. Exportar destaque
top_ganhos_por_y.to_csv('top_aditivos_por_alvo.csv', index=False, encoding='utf-8-sig')

Processando ABE_ESP...
Processando ABC_ESP...
Processando ABB_ESP...
Processando ABI_ESP...

--- RESUMO DE ADITIVOS ESTATISTICAMENTE SIGNIFICATIVOS ---
   Alvo (y) Aditivo  Coeficiente (Ganho/Perda)  P-Valor Tendência
0   ABE_ESP       C                  -324.5999   0.0177  Negativa
1   ABE_ESP       D                  -686.6510   0.0000  Negativa
2   ABE_ESP       E                 -1786.5011   0.0000  Negativa
3   ABC_ESP       E                     0.1965   0.0002  Positiva
4   ABC_ESP       I                    -0.1105   0.0129  Negativa
5   ABB_ESP       B                     0.1178   0.0169  Positiva
6   ABB_ESP       C                     0.1733   0.0003  Positiva
7   ABB_ESP       D                     0.2522   0.0000  Positiva
8   ABB_ESP       E                     0.4085   0.0000  Positiva
9   ABB_ESP       I                     0.1229   0.0097  Positiva
10  ABI_ESP       B                     1.2728   0.0051  Positiva
11  ABI_ESP       C                     1.8486   0.0000 